In [20]:
# ----------------------------
# PACE matchups
# ----------------------------
# Reads ./data/lsu_hypoxia_2024.csv
# Searches PACE_OCI_L2_BGC within ±3 hours for each station
# Chooses the granule closest in time
# Extracts chlor_a (nearest valid pixel + 3×3 median)
# Writes ./output/pace_matchups.csv

from __future__ import annotations

import os
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import earthaccess
import xarray as xr

try:
    from datatree import open_datatree  # type: ignore
except Exception:
    open_datatree = None

In [24]:
# ---------------------------
# Config
# ---------------------------
DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")

INPUT_CSV = DATA_DIR / "lsu_hypoxia_2024.csv"
OUTPUT_CSV = OUTPUT_DIR / "pace_matchups.csv"

TIME_WINDOW_HOURS = 24
BBOX_HALF_WIDTH_DEG = 0.25

PACE_SHORTNAMES = [
    "PACE_OCI_L2_BGC",
    "PACE_OCI_L2_BGC_NRT",
]

CHL_CANDIDATE_NAMES = ["chlor_a", "chl", "chlorophyll_a", "Chlorophyll_a"]
FLAG_CANDIDATE_NAMES = ["l2_flags", "flags", "quality_flags", "qual_flags"]



In [22]:
# ---------------------------
# Helpers
# ---------------------------
def ensure_dirs() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def parse_field_datetime_utc(date_str: str, time_str: str) -> datetime:
    dt = datetime.strptime(f"{date_str} {time_str}", "%m/%d/%Y %H:%M")
    return dt.replace(tzinfo=timezone.utc)


def station_bbox(lat: float, lon: float, half_width_deg: float) -> Tuple[float, float, float, float]:
    return (lon - half_width_deg, lat - half_width_deg, lon + half_width_deg, lat + half_width_deg)


def haversine_km(lat1: float, lon1: float, lat2: np.ndarray, lon2: np.ndarray) -> np.ndarray:
    r = 6371.0
    lat1r = np.deg2rad(lat1)
    lon1r = np.deg2rad(lon1)
    lat2r = np.deg2rad(lat2)
    lon2r = np.deg2rad(lon2)
    dlat = lat2r - lat1r
    dlon = lon2r - lon1r
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1r) * np.cos(lat2r) * np.sin(dlon / 2.0) ** 2
    return 2.0 * r * np.arcsin(np.sqrt(a))


def safe_get_attr_time(ds: xr.Dataset, key: str) -> Optional[datetime]:
    val = ds.attrs.get(key)
    if val is None:
        return None
    s = str(val).strip()
    if s.endswith("Z"):
        s = s[:-1] + "+00:00"
    try:
        dt = datetime.fromisoformat(s)
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt.astimezone(timezone.utc)
    except Exception:
        return None


def granule_mid_time_utc(ds: xr.Dataset) -> Optional[datetime]:
    t0 = safe_get_attr_time(ds, "time_coverage_start")
    t1 = safe_get_attr_time(ds, "time_coverage_end")
    if t0 and t1:
        return t0 + (t1 - t0) / 2
    return t0 or t1


def find_var_name(ds: xr.Dataset, candidates: List[str]) -> Optional[str]:
    for name in candidates:
        if name in ds.variables:
            return name
    lowered = {v.lower(): v for v in ds.variables}
    for cand in candidates:
        if cand.lower() in lowered:
            return lowered[cand.lower()]
    return None


def open_pace_l2_as_dataset(file_obj: Any) -> xr.Dataset:
    if open_datatree is not None:
        try:
            dt = open_datatree(file_obj)
            datasets: List[xr.Dataset] = []
            try:
                datasets.append(dt.ds)
            except Exception:
                pass
            for node in dt.subtree:
                try:
                    if hasattr(node, "ds") and isinstance(node.ds, xr.Dataset):
                        datasets.append(node.ds)
                except Exception:
                    continue
            return xr.merge(datasets, compat="override", join="outer")
        except Exception:
            pass

    for engine in ("h5netcdf", "netcdf4"):
        try:
            return xr.open_dataset(file_obj, engine=engine)
        except Exception:
            continue

    return xr.open_dataset(file_obj)


def extract_nearest_and_3x3(
    ds: xr.Dataset,
    station_lat: float,
    station_lon: float,
    chl_var: str,
    flag_var: Optional[str],
) -> Dict[str, Any]:
    lat_name = find_var_name(ds, ["latitude", "lat", "Latitude"])
    lon_name = find_var_name(ds, ["longitude", "lon", "Longitude"])
    if lat_name is None or lon_name is None:
        raise ValueError("Latitude/longitude variables not found in this granule.")

    lat2d = ds[lat_name].values
    lon2d = ds[lon_name].values
    chl = ds[chl_var].values

    chl_valid = np.isfinite(chl)

    if np.any(chl_valid):
        latv = lat2d[chl_valid]
        lonv = lon2d[chl_valid]
        dv = haversine_km(station_lat, station_lon, latv, lonv)
        k = int(np.argmin(dv))
        i, j = np.argwhere(chl_valid)[k]
        nearest_dist_km = float(dv[k])
    else:
        d_all = haversine_km(station_lat, station_lon, lat2d, lon2d)
        i, j = np.unravel_index(int(np.argmin(d_all)), d_all.shape)
        nearest_dist_km = float(d_all[i, j])

    nearest_chl = float(chl[i, j]) if np.isfinite(chl[i, j]) else np.nan

    i0, i1 = max(i - 1, 0), min(i + 2, chl.shape[0])
    j0, j1 = max(j - 1, 0), min(j + 2, chl.shape[1])
    window = chl[i0:i1, j0:j1]

    med_3x3 = float(np.nanmedian(window)) if np.any(np.isfinite(window)) else np.nan
    n_valid_3x3 = int(np.sum(np.isfinite(window)))

    raw_flag = None
    if flag_var and flag_var in ds.variables:
        try:
            raw_flag = int(ds[flag_var].values[i, j])
        except Exception:
            raw_flag = None

    return {
        "pace_nearest_chlor_a": nearest_chl,
        "pace_3x3_median_chlor_a": med_3x3,
        "pace_nearest_pixel_lat": float(lat2d[i, j]),
        "pace_nearest_pixel_lon": float(lon2d[i, j]),
        "pace_nearest_pixel_distance_km": nearest_dist_km,
        "pace_raw_flag_nearest": raw_flag,
        "pace_pixel_valid_basic": bool(np.isfinite(nearest_chl)),
        "pace_n_valid_in_3x3": n_valid_3x3,
    }


def get_granule_id(res: Any) -> Optional[str]:
    for attr in ("producer_granule_id", "title", "name"):
        try:
            if hasattr(res, attr):
                v = getattr(res, attr)
                if v:
                    return str(v)
        except Exception:
            pass
    try:
        for key in ("producer_granule_id", "title", "name"):
            v = res.get(key, None)
            if v:
                return str(v)
    except Exception:
        pass
    return None


def search_pace_granules(field_time_utc: datetime, lat: float, lon: float) -> List[Any]:
    start = field_time_utc - timedelta(hours=TIME_WINDOW_HOURS)
    end = field_time_utc + timedelta(hours=TIME_WINDOW_HOURS)
    bbox = station_bbox(lat, lon, BBOX_HALF_WIDTH_DEG)

    for sn in PACE_SHORTNAMES:
        try:
            results = list(
                earthaccess.search_data(
                    short_name=sn,
                    temporal=(start.isoformat(), end.isoformat()),
                    bounding_box=bbox,
                )
            )
            if results:
                return results
        except Exception:
            continue
    return []



In [25]:
# ---------------------------
# Run
# ---------------------------
ensure_dirs()

if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Expected input CSV at: {INPUT_CSV.resolve()}")

# Login via env vars set in your environment (PyCharm, terminal, etc.)
if not os.environ.get("EARTHDATA_USERNAME") or not os.environ.get("EARTHDATA_PASSWORD"):
    raise RuntimeError("Missing EARTHDATA_USERNAME/EARTHDATA_PASSWORD in environment.")

earthaccess.login(strategy="environment")

field = pd.read_csv(INPUT_CSV, encoding="ISO-8859-1").copy()

col_date = "Date"
col_time = "Time CTD Cast began (UTC)"
col_station = "Station"
col_lat = "Lat"
col_lon = "Long"
col_do = "Corrected DO (mg/L)"
col_sal = "Surface Sal (psu)"
col_chl = "Surface Chl ( µg/L)"

field["field_time_utc"] = [
    parse_field_datetime_utc(d, t)
    for d, t in zip(field[col_date].astype(str), field[col_time].astype(str))
]

rows: List[Dict[str, Any]] = []

for _, r in field.iterrows():
    station = str(r[col_station])
    lat = float(r[col_lat])
    lon = float(r[col_lon])
    ft = r["field_time_utc"]

    base = {
        "Station": station,
        "field_time_utc": ft.isoformat(),
        "field_lat": lat,
        "field_lon": lon,
        "field_surface_chl_ugL": float(r[col_chl]),
        "field_surface_sal_psu": float(r[col_sal]),
        "field_corrected_do_mgL": float(r[col_do]),
    }

    results = search_pace_granules(ft, lat, lon)
    if not results:
        rows.append({**base, "pace_found": False, "error": None})
        continue

    best_row: Optional[Dict[str, Any]] = None
    best_abs_offset_min = float("inf")
    last_error: Optional[str] = None

    for res in results:
        try:
            files = list(earthaccess.open([res]))
            if not files:
                last_error = "earthaccess.open returned no files"
                continue

            ds = open_pace_l2_as_dataset(files[0])

            chl_var = find_var_name(ds, CHL_CANDIDATE_NAMES)
            if chl_var is None:
                last_error = "chlor_a variable not found"
                continue

            flag_var = find_var_name(ds, FLAG_CANDIDATE_NAMES)

            tmid = granule_mid_time_utc(ds)
            if tmid is None:
                offset_min = None
                abs_offset = float("inf")
            else:
                offset_min = (tmid - ft).total_seconds() / 60.0
                abs_offset = abs(offset_min)

            ex = extract_nearest_and_3x3(ds, lat, lon, chl_var, flag_var)

            candidate = {
                **base,
                "pace_found": True,
                "pace_granule": get_granule_id(res),
                "pace_time_utc": tmid.isoformat() if tmid else None,
                "pace_time_offset_minutes": float(offset_min) if offset_min is not None else None,
                **ex,
                "error": None,
            }

            if abs_offset < best_abs_offset_min:
                best_abs_offset_min = abs_offset
                best_row = candidate

        except Exception as e:
            last_error = str(e)
            continue

    if best_row is not None:
        rows.append(best_row)
    else:
        rows.append({**base, "pace_found": False, "error": last_error})

out = pd.DataFrame(rows)
out.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote: {OUTPUT_CSV.resolve()}")
out.head(10)

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

...

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

Wrote: /home/jovyan/shared-public/Hypoxia_Group/output/pace_matchups.csv


,Station,field_time_utc,field_lat,field_lon,field_surface_chl_ugL,field_surface_sal_psu,field_corrected_do_mgL,pace_found,error
0,A0,2024-07-22T12:57:00+00:00,29.2962,-89.7498,23.51,16.43,3.16,False,chlor_a variable not found
1,A1,2024-07-22T12:32:00+00:00,29.2886,-89.7498,21.73,16.96,0.50,False,chlor_a variable not found
2,A2,2024-07-22T11:48:00+00:00,29.2393,-89.7500,19.64,18.06,0.33,False,chlor_a variable not found
3,A3,2024-07-22T10:30:00+00:00,29.1770,-89.7505,15.49,22.11,1.08,False,chlor_a variable not found
4,A4,2024-07-22T09:47:00+00:00,29.1326,-89.7499,1.61,25.48,1.67,False,chlor_a variable not found
5,A5,2024-07-22T08:53:00+00:00,29.0700,-89.7500,1.38,25.88,3.26,False,chlor_a variable not found
6,A6,2024-07-22T07:52:00+00:00,29.0083,-89.7500,1.69,25.28,3.42,False,chlor_a variable not found
7,AA1,2024-07-21T23:43:00+00:00,29.1348,-89.4754,15.24,14.05,0.23,False,chlor_a variable not found
8,AA2,2024-07-22T01:02:00+00:00,29.0942,-89.4976,4.80,21.57,0.29,False,chlor_a variable not found
9,AA3,2024-07-22T02:13:00+00:00,29.0338,-89.5330,9.17,23.62,3.42,False,chlor_a variable not found


In [6]:
import os
import getpass
import earthaccess

os.environ["EARTHDATA_USERNAME"] = input("Earthdata username: ").strip()
os.environ["EARTHDATA_PASSWORD"] = getpass.getpass("Earthdata password: ")

earthaccess.login(strategy="environment")

Earthdata username:  ralfriedel
Earthdata password:  ········


In [10]:
# Search dataset catalog for likely matches
import earthaccess
earthaccess.login(strategy="environment")

ds = list(earthaccess.search_datasets(keyword="PACE OCI L2 OC"))
print("Datasets:", len(ds))
for d in ds[:25]:
    # Different earthaccess versions expose different fields; this is best-effort
    try:
        print(d.get("short_name"), "-", d.get("title"))
    except Exception:
        print(d)


Datasets: 28
None - None
None - None
None - None
None - None
None - None
None - None
None - None
None - None
None - None
None - None
None - None
None - None
None - None
None - None
None - None
None - None
None - None
None - None
None - None
None - None
None - None
None - None
None - None
None - None
None - None


In [12]:
#Hits = 0 bboxes wrong
lat = 29.0
lon = -90.0
bbox = (lon-1.0, lat-1.0, lon+1.0, lat+1.0)  # 1 degree half-width

results = list(
    earthaccess.search_data(
        short_name="(your working short_name)",
        temporal=("2024-07-22T00:00:00Z", "2024-07-23T00:00:00Z"),
        bounding_box=bbox,
    )
)
print("Hits with bbox:", len(results))


Hits with bbox: 0


In [9]:
#Hits = 0 short_name is not correct
#Hits > 0 bounding boxes need adjusting
import earthaccess

earthaccess.login(strategy="environment")

#pick any date for diagnostic
results = list(
    earthaccess.search_data(
        short_name="PACE_OCI_L2_OC_BGC",
        temporal=("2024-07-22T00:00:00Z", "2024-07-23T00:00:00Z"),
        # no bounding box; testing only
    )
)

print("Hits:", len(results))
if results:
    print(results[0])

Hits: 0


In [17]:
#still nothing found
#code above is grabbing a collection of objects, not dicts
#this fixes that:

import earthaccess

earthaccess.login(strategy="environment")

ds = list(earthaccess.search_datasets(keyword="PACE OCI L2 OC"))
print("Datasets:", len(ds))

for i, d in enumerate(ds[:50]):
    umm = getattr(d, "umm", None)
    if isinstance(umm, dict):
        short = umm.get("ShortName")
        title = umm.get("EntryTitle") or umm.get("Abstract")
        prov = umm.get("Provider")  # sometimes present
        ver = umm.get("Version")    # sometimes present
        #print(f"{i:02d}  ShortName={short}  Version={ver}  Provider={prov}  Title={title}")
    else:
        # fallback: at least show object repr
        #print(f"{i:02d}  {d!r}")
        print('something')


Datasets: 28
something
something
something
something
something
something
something
something
something
something
something
something
something
something
something
something
something
something
something
something
something
something
something
something
something
something
something
something


In [18]:
ds = list(earthaccess.search_datasets(keyword="PACE OCI L2 OC_BGC"))
print("Datasets:", len(ds))
for d in ds:
    umm = d.umm
    print(umm.get("ShortName"), "-", umm.get("EntryTitle"))


Datasets: 1


AttributeError: 'DataCollection' object has no attribute 'umm'

In [19]:
import earthaccess

earthaccess.login(strategy="environment")

ds = list(earthaccess.search_datasets(keyword="PACE OCI L2"))
print("Datasets:", len(ds))

for i, d in enumerate(ds[:10]):
    print("\n---", i, "---")
    print("type:", type(d))
    print("repr:", d)

    # Try common locations
    umm = getattr(d, "umm", None)
    if isinstance(umm, dict):
        print("umm keys:", sorted(umm.keys())[:50])
        print("ShortName:", umm.get("ShortName"))
        print("EntryTitle:", umm.get("EntryTitle"))
        continue

    # Some versions expose a 'data' dict
    data = getattr(d, "data", None)
    if isinstance(data, dict):
        print("data keys:", sorted(data.keys())[:50])
        if "umm" in data and isinstance(data["umm"], dict):
            print("ShortName:", data["umm"].get("ShortName"))
            print("EntryTitle:", data["umm"].get("EntryTitle"))
        continue

    # Last resort: dump attribute names so we can spot it
    print("attrs:", [a for a in dir(d) if ("short" in a.lower() or "umm" in a.lower() or "title" in a.lower())])


Datasets: 29

--- 0 ---
type: <class 'earthaccess.results.DataCollection'>
repr: {
  "meta": {
    "revision-id": 2,
    "deleted": false,
    "format": "application/vnd.nasa.cmr.umm+json",
    "provider-id": "OB_CLOUD",
    "has-combine": false,
    "user-id": "amscott",
    "has-formats": true,
    "associations": {
      "variables": [
        "V3797607869-OB_CLOUD",
        "V3797608190-OB_CLOUD",
        "V3797607910-OB_CLOUD",
        "V3797607888-OB_CLOUD",
        "V3797607867-OB_CLOUD",
        "V3797608167-OB_CLOUD",
        "V3797607937-OB_CLOUD",
        "V3797608180-OB_CLOUD",
        "V3797607900-OB_CLOUD",
        "V3797608215-OB_CLOUD",
        "V3797607960-OB_CLOUD",
        "V3797608038-OB_CLOUD",
        "V3797608020-OB_CLOUD",
        "V3797608193-OB_CLOUD",
        "V3797607857-OB_CLOUD",
        "V3797608080-OB_CLOUD",
        "V3797608205-OB_CLOUD",
        "V3797608062-OB_CLOUD",
        "V3797608003-OB_CLOUD",
        "V3797607875-OB_CLOUD",
        "V379760786